# R2 — ResNet-18 Double Descent

| | |
|---|---|
| Model | WideResNet18 (channels = [k, 2k, 4k, 8k]) |
| Dataset | CIFAR-10, n=5000, η=15% |
| Sweep | k ∈ {1,2,4,8,16,32} × 1 seed = **6 runs** |
| Optimizer | SGD + cosine LR decay, 200 epochs |
| Output | `fig2_r2_dd.png` — Figure 2 with interpolation threshold + param-count axis |

**Estimated time on T4: ~3–4 hours total.**


## Step 1 — Environment

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Drive + Clone repo

In [ ]:
import os, sys
from google.colab import drive

TOKEN      = 'ghp_你的token'   # ← replace with your PAT
REPO_DIR   = '/content/project-6699'
RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/R2'

drive.mount('/content/drive')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

REPO_URL = f'https://{TOKEN}@github.com/alice20030504/EECS-6699.git'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch yixuan {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} checkout yixuan')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Ready. Files:', [f for f in os.listdir('.') if f.endswith('.py')])

## Step 3 — Run R2 (6 runs, auto-skips completed ones)

In [ ]:
from run_r2 import R2_CONFIG, run_r2, plot_r2
import copy, threading, time

def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js; eval_js('0')
        except: pass
threading.Thread(target=_keep_alive, daemon=True).start()

cfg = copy.deepcopy(R2_CONFIG)
results = run_r2(cfg, RESULT_DIR, resume=True)
print(f'Done. {len(results)} runs.')

## Step 4 — Plot Figure 2

Generates Fig 2 with interpolation threshold line and parameter-count axis.

**Can be re-run anytime without retraining.**

In [ ]:
from run_r2 import plot_r2
plot_r2(RESULT_DIR)

from IPython.display import Image, display
from pathlib import Path
p = Path(RESULT_DIR) / 'fig2_r2_dd.png'
if p.exists(): display(Image(str(p)))

## Step 5 — Summary table

In [ ]:
import pandas as pd
from src.io_utils import load_results

results = load_results(RESULT_DIR, pattern='r2_*.json')
df = pd.DataFrame([{
    'k':         r['width_multiplier'],
    'seed':      r['seed'],
    'n_params':  r['n_params'],
    'train_err': f"{r['train_error']:.3f}",
    'test_err':  f"{r['test_error']:.3f}",
    'time_min':  f"{r['wall_time_s']/60:.1f}",
} for r in results]).sort_values(['k', 'seed'])
print(df.to_string(index=False))